# 03 — Schematise

`octi` snaps the network onto a grid. This is the step that turns a geographic
map into a tube map.

**Measure octilinearity on a projected graph.** LOOM computes in Web Mercator
metres but writes lon/lat, so at LA's latitude a true 45° edge reads as about
39° — the map looks broken when it is not.

In [ ]:
%load_ext autoreload
%autoreload 2

from schematic import feeds, loom, pipeline, animate
from schematic.linegraph import LineGraph
from schematic.crs import to_mercator
from schematic.render import render, octilinearity, Style

FEED = "la-metro-rail"
LINE_ORDER = list("ABCDEK")   # the order lines are drawn in, back to front

In [ ]:
paths = pipeline.schematize(FEED)
before = LineGraph.from_geojson(paths["loom"]).reproject(to_mercator)
after = LineGraph.from_geojson(paths["octi"]).reproject(to_mercator)

for name, g in [("geographic", before), ("schematic", after)]:
    ok, total = octilinearity(g)
    print(f"{name:12} {100*ok/total:5.1f}% of drawn length on a 45-degree multiple")

### Why the metric is weighted by length

LOOM writes coordinates at six decimal places, which leaves a scatter of
metre-scale stubs around station nodes. Their angles are pure rounding noise,
and counting segments lets them dominate a number that should describe what you
can see.

In [ ]:
import math
from schematic.offsets import dedupe

devs = [(min(a := math.degrees(math.atan2(q[1]-p[1], q[0]-p[0])) % 45, 45-a), math.dist(p, q))
        for e in after.edges for p, q in zip(dedupe(list(e.geometry)), dedupe(list(e.geometry))[1:])]
short = [d for d, L in devs if L < 30]
print(f"{len(short)}/{len(devs)} segments are under 30 m")
print(f"unweighted: {100*sum(1 for d, _ in devs if d <= 1)/len(devs):.1f}%")
print(f"by length:  {100*octilinearity(after)[0]/octilinearity(after)[1]:.1f}%")

### Trying other base graphs

`octi -b` also does `orthoradial` (concentric rings, like Moscow's map),
`ortholinear` (90° only) and `hexalinear`.

In [ ]:
import json
payload = paths["loom"].read_bytes()
for base in ["octilinear", "orthoradial", "ortholinear"]:
    g = LineGraph.from_geojson(loom.run("octi", payload, "-b", base)).reproject(to_mercator)
    ok, total = octilinearity(g)
    print(f"{base:12} {len(g.nodes):4} nodes  {100*ok/total:5.1f}% octilinear")